# Evidence-plus-answer SFT preparation — alias occurrence only

**Run All defaults to DEV ONLY.** In the first setup cell, `ANNOTATION_SPLITS = ["dev"]` skips every training configuration, API call, preview and export cell. Select `["train"]` for training annotation, or `["train", "dev"]` for both. After changing the choice, rerun setup. Existing training caches/exports are preserved.


**No LLM, server, API key or GPU required.** Run all cells with the `bioasq` Python kernel.

This alternative to `prepare_factoid_evidence_answer_sft.ipynb` numbers individual snippets, searches each for every gold alias, and constructs targets such as:

```text
Evidence: [2, 5]
Answer: [BE]15[EE]
```

The evidence IDs mean **the selected answer alias occurs in these snippets**. They are weak supervision, not verified semantic support. An absent alias is recorded as `no_occurrence_match`, not semantically wrong or unsupported. No semantic judge is invoked.

The input pool is 1,130 training questions. All 1,463 original prepared gold aliases are audited, including aliases the previous 1,352-row occurrence filter omitted. Known question and matching-alias flags remain excluded automatically. This notebook prepares datasets; it does not train a model.

**New:** the final dev-only section exports reusable occurrence labels for all 160 dev questions. Semantic support is added in the judge notebook, not inferred here.


In [ ]:
# Run All processes ONLY these splits. Change explicitly to ["train"] or ["train", "dev"].
ANNOTATION_SPLITS = ["dev"]
assert ANNOTATION_SPLITS and set(ANNOTATION_SPLITS) <= {"train", "dev"}
print("Enabled annotation splits:", ANNOTATION_SPLITS)

from pathlib import Path
from datetime import datetime, timezone
import sys
import json


def find_project_root(start):
    for parent in [start, *start.parents]:
        for candidate in [parent, parent / "Task-Structured Counterfactual Preference Mining"]:
            if (candidate / "cse_dpo").is_dir() and (candidate / "src").is_dir():
                return candidate.resolve()
    raise RuntimeError("Open the notebook within the BioASQ_MRES workspace")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Reuse only source loading, hashes and file helpers from the first preparation pipeline.
from cse_dpo.evidence_sft_data import build_packets, read_jsonl, file_hash
from cse_dpo import occurrence_sft_data as occurrence
print("Project:", PROJECT_ROOT)


## Settings

- `normalized` uses the project's existing alias normalization and complete normalized-token matching. It tolerates some punctuation, case and spacing differences; normalization can conflate biomedical expressions, so the labels remain heuristic.
- `literal` uses case-sensitive exact text with word boundaries. It may miss valid spelling variants.
- With `EXPORT_ALL_MATCHING_ALIASES=True`, each eligible alias receives its own example and its own matching snippet IDs. False exports the first eligible original alias per question. Both arms use the identical selection; all aliases always remain in the audit.

Snippet order is preserved. All matching snippet IDs are used; there is no top-k cap, no semantic ranking, and no removal of distracting snippets. Duplicate snippet occurrences retain distinct IDs. Keep all variants of a question in the same data split.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    MATCH_MODE = "normalized"  # "normalized" or "literal"
    EXPORT_ALL_MATCHING_ALIASES = True
    RUN_NAME = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    OUTPUT_DIR = PROJECT_ROOT / "Artifacts/Factoid_SFT/evidence_occurrence" / RUN_NAME
    assert MATCH_MODE in {"normalized", "literal"}
    print("Output:", OUTPUT_DIR)
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## Load the complete contexts and original aliases

The shared loader restricts the original training data to the supported-pool IDs, checks dev separation, and imports prior exclusions. It does not call a judge or use new semantic labels.

Each `[BS]…[ES]` span is an individual snippet. The notebook retains the resource's PubMed ID, source field, local snippet index, original text, and character offsets. Labels are inserted before snippet boundaries; surrounding text, metadata, duplicates and nonmatching snippets remain in the prompt. Matching searches only snippet bodies, so numbers in PubMed IDs or section metadata cannot count as answer hits.

Resources with no markers fall back to their full body, with the PMID header excluded from matching. Malformed marker sequences hold the question for inspection; they are never silently truncated or repaired. No sentence segmentation is inferred inside a supplied snippet.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    packets, source_manifest = build_packets(PROJECT_ROOT)
    assert len(packets) == 1130, "Source pool changed; inspect the count before continuing."
    print("Questions:", len(packets))
    print("Original aliases:", sum(len(p["aliases"]) for p in packets))
    print("Previously flagged questions:", source_manifest["previously_flagged_questions"])
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    # Preview one real example before export.
    example = next(p for p in packets if not p["prior_question_flag"])
    labeled = occurrence.label_packet(example, MATCH_MODE)
    print(example["question_id"], example["question"])
    print("Numbered snippets:", len(labeled["snippets"]))
    for alias in labeled["aliases"]:
        print(repr(alias["alias"]), alias["occurrence_status"], alias["matching_snippet_ids"],
              "excluded:", alias["exclusion_reasons"])
    first_eligible = next((a for a in labeled["aliases"] if a["eligible_for_sft"]), None)
    if first_eligible:
        preview = occurrence.make_record(example, labeled, first_eligible, evidence_first=True)
        print("Training target:\n" + preview["messages"][-1]["content"])
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## Export both training arms and the complete alias audit

All known unresolved question flags and corresponding alias flags remain excluded, even when their text matches. This mode cannot discover or resolve new semantic ambiguity. Previously flagged examples stay visible in the reports.

These are chat-message SFT records. The evidence-plus-answer target includes both lines; a later training notebook must preserve both, rather than pass them through the old answer-only target parser. During evaluation, score only the `Answer:` line with BioASQ metrics and assess occurrence prediction separately. Use a training context limit covering the entire input and output; evidence IDs must never refer to truncated snippets.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    summary = occurrence.export_occurrence_data(
        packets, source_manifest, OUTPUT_DIR,
        mode=MATCH_MODE, all_aliases=EXPORT_ALL_MATCHING_ALIASES,
    )
    print(json.dumps({k: v for k, v in summary.items()
                      if k not in {"source_manifest", "output_hashes"}}, indent=2))
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## Verify preserved contexts and matching targets

Check exact snippet offsets, all predicted evidence IDs, identical inputs/answer targets across the two arms, and source hashes. This validates the data construction, not semantic support.


In [ ]:
if "train" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    by_id = {p["question_id"]: p for p in packets}
    evidence_rows = read_jsonl(OUTPUT_DIR / "evidence_sft.jsonl")
    answer_rows = read_jsonl(OUTPUT_DIR / "answer_only_sft.jsonl")
    assert len(evidence_rows) == len(answer_rows)
    for evidence, answer in zip(evidence_rows, answer_rows):
        assert evidence["id"] == answer["id"]
        assert evidence["messages"][1] == answer["messages"][1]
        assert evidence["metadata"]["answer"] == answer["metadata"]["answer"]

    indexes = {}
    for s in read_jsonl(OUTPUT_DIR / "snippet_index.jsonl"):
        original = next(r["text"] for r in by_id[s["question_id"]]["resources"] if r["resource_id"] == s["resource_id"])
        assert original[s["body_start"]:s["body_end"]] == s["text"]
        indexes.setdefault(s["question_id"], {})[s["snippet_id"]] = s
    aliases = read_jsonl(OUTPUT_DIR / "alias_occurrences.jsonl")
    assert len(aliases) == sum(len(p["aliases"]) for p in packets)
    for alias in aliases:
        expected_ids = [i for i, s in indexes.get(alias["question_id"], {}).items()
                        if occurrence.contains_alias(alias["alias"], s["text"], MATCH_MODE)]
        assert alias["matching_snippet_ids"] == expected_ids
        if alias["eligible_for_sft"]:
            assert expected_ids and not alias["prior_question_flag"] and not alias["prior_alias_flags"]
    for path, expected_hash in source_manifest["source_hashes"].items():
        assert file_hash(path) == expected_hash, f"Source changed: {path}"
    print("PASS: complete alias audit, matching snippet IDs, original snippet text and matched training arms.")
    print("Dataset directory:", OUTPUT_DIR)
else:
    print("Skipping train section (disabled in ANNOTATION_SPLITS).")


## Files produced

- **`evidence_sft.jsonl`**: full numbered contexts → matching snippet IDs + answer.
- **`answer_only_sft.jsonl`**: identical contexts and aliases → answer only.
- **`alias_occurrences.csv` / `.jsonl`**: every alias, occurrence status, matching IDs, matching snippet text, PubMed IDs and exclusion reasons.
- **`question_summary.jsonl`**: matched/unmatched aliases and export status per question.
- **`snippet_index.jsonl`**: snippet number → exact text, resource, PMID and offsets.
- **`source_packets.jsonl`**: original full contexts and provenance.
- **`exclusions.jsonl`**: no-match aliases, known flags and parsing issues.
- **`summary.json`**: counts, matching policy and source/output hashes.

This is a baseline for **answer-occurrence supervision**. It does not claim that matching snippets establish the question's relationship, that nonmatching snippets are irrelevant, or that the model learns faithful evidence use. Evaluate those questions separately from exact-match answer accuracy.


## Dev occurrence labels (evaluation only)

Run All defaults to dev only; training exports are skipped. You can also run setup then this section. It does not run the training export or call a model. Number **all 160 dev questions** identically to the shared evidence-plus-answer evaluator, retain every accepted alias, and record literal and normalized matches for every alias–snippet pair.

This uses the same frozen dev pool as the semantic preparation notebook. `support` stays `unreviewed`: occurrence is not evidence correctness. The semantic notebook adds training-style sufficient evidence sets separately; uncited snippets remain unreviewed. Neither notebook emits SFT examples from dev data.


In [ ]:
if "dev" in globals().get("ANNOTATION_SPLITS", ["dev"]):
    import importlib
    from cse_dpo import dev_evidence_annotation as devprep
    devprep = importlib.reload(devprep)
    DEV_RUN_DIR = PROJECT_ROOT / "Artifacts/Factoid_SFT/dev_evidence_annotation/full_dev_evidence_train_aligned_v2"
    dev_packets, dev_manifest = devprep.build_packets(PROJECT_ROOT)
    assert dev_manifest["question_count"] == 160 and dev_manifest["train_dev_disjoint"]
    devprep.initialize_run(DEV_RUN_DIR, dev_packets, dev_manifest)
    DEV_OCCURRENCE_EXPORT = DEV_RUN_DIR / "occurrence_exports" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    dev_occurrence_summary = devprep.export_annotations(
        dev_packets, dev_manifest, {}, DEV_OCCURRENCE_EXPORT, occurrence_only=True,
    )
    print(json.dumps({k:v for k,v in dev_occurrence_summary.items() if k not in {"source_manifest", "output_hashes"}}, indent=2))
    print("Dev occurrence reference:", DEV_OCCURRENCE_EXPORT)
    assert not (DEV_OCCURRENCE_EXPORT / "evidence_sft.jsonl").exists()
    assert not (DEV_OCCURRENCE_EXPORT / "answer_only_sft.jsonl").exists()
else:
    print("Skipping dev section (disabled in ANNOTATION_SPLITS).")
